# Step 3 evidence: reproduce every number quoted during data checks

Each section below reproduces a finding from the Step 3 discussion, from the **raw** files (read-only).
Run top to bottom. If a number differs from the expected one in the check line, tell Claude before relying on it.

## E0. Raw-file integrity (run this first, and any time you suspect a file was touched)

In [ ]:
import os
import pandas as pd

RAW = os.path.join('..', 'data-pack', 'data')
EXPECTED = dict(sku_master=(36,'sku_code'), stockist_master=(156,'stockist_code'), batch_master=(663,'batch_no'),
                primary_sales=(93809,'invoice_no'), expiry_claims=(4526,'claim_no'),
                cnf_stock_snapshot=(5015,'snapshot_date'), claim_emails=(30,'email_id'))

def load(n): return pd.read_csv(os.path.join(RAW, n + '.csv'), dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
R = {n: load(n) for n in EXPECTED}

for n, (rows, first_col) in EXPECTED.items():
    with open(os.path.join(RAW, n + '.csv'), encoding='utf-8') as fh: header = fh.readline().strip()
    ok_header = header.split(',')[0] == first_col          # header must be line 1 (a sort can push it to the bottom)
    ok_rows = len(R[n]) == rows
    print(f'{n:22s} header_first={ok_header}  rows={len(R[n])} (expected {rows})  {"OK" if ok_header and ok_rows else "<<< CHECK THIS FILE"}')
# primary_sales invoice INV/E01/2404/000889 must have a BLANK scheme_code in the original
ps0 = R['primary_sales']
print('INV/E01/2404/000889 scheme_code =', repr(ps0.loc[ps0.invoice_no == 'INV/E01/2404/000889', 'scheme_code'].iloc[0]), '(original is empty)')

**Check:** every line says `OK`, and the last line shows `''` (empty). If not, restore the file from git (`git restore data-pack/data/<file>.csv`) before doing anything else.

## E1. Blank `original_invoice_ref` (expiry_claims): what it means

In [ ]:
ec = R['expiry_claims']
blank = ec.original_invoice_ref == ''
print('blank invoice ref:', int(blank.sum()), 'of', len(ec), f'({blank.mean()*100:.1f}%)')
print('\nby status:\n', ec[blank].status.value_counts().to_string())
print('\nremarks on these rows:\n', ec[blank].remarks.replace('', '(no remark)').value_counts().to_string())
print('\nBlank-ref rows by claim_type:\n', ec[blank].claim_type.value_counts().to_string())
print('\nColumns that could hold Area Manager approval:', [c for c in ec.columns if 'approv' in c.lower() or 'manager' in c.lower()] or 'NONE')

**Check:** 884 blank (19.5%). Credited 802, Pending 64, Rejected 18. Remarks: 286 / 258 / 258 / 18 / 64 with none. No approval column exists, so R2's second half is untestable in the data.

## E2. Blank `credit_note_no`: it follows the status

In [ ]:
cn_blank = ec.credit_note_no == ''
print(pd.crosstab(ec.status, cn_blank.map({True: 'credit note blank', False: 'has credit note'})))
print('\nRejected claims, remarks:', ec[ec.status == 'Rejected'].remarks.value_counts().to_dict())

**Check:** blank = Pending 366 + Rejected 31 = 397. All 4,129 Credited have a number. All 31 Rejected say `Duplicate of earlier claim`.

## E3. Blank `ordered_qty` (primary_sales): invoice side is still complete

In [ ]:
ps = R['primary_sales'].copy()
num = lambda s: pd.to_numeric(s.str.replace(',', '', regex=False), errors='coerce')
for c in ['ordered_qty','invoiced_qty','free_qty','pts_inr','net_value_inr']: ps[c] = num(ps[c])
ob = ps.ordered_qty.isna()
print('blank ordered_qty:', int(ob.sum()), f'({ob.mean()*100:.1f}%)   invoiced_qty blank among them:', int(ps[ob].invoiced_qty.isna().sum()))
print('net_value == invoiced_qty x pts (within 1 INR) on blank rows:', f'{((ps[ob].invoiced_qty*ps[ob].pts_inr - ps[ob].net_value_inr).abs() < 1).mean()*100:.1f}%')
ok = ps[~ob]
print('rows with an order: ordered > invoiced:', int((ok.ordered_qty > ok.invoiced_qty).sum()), f'({(ok.ordered_qty > ok.invoiced_qty).mean()*100:.1f}%); ordered < invoiced:', int((ok.ordered_qty < ok.invoiced_qty).sum()))
print('\nblank ordered_qty by C&F:\n', ps[ob].cnf_code.value_counts().to_string())

**Check:** 3,069 blank (3.3%); net value matches on 100.0%; ordered > invoiced on 21,223 rows (23.4% of 90,740) and never below.

## E4. What `scheme_code` blank / `NA` / `-` mean (inference: all three = no scheme)

In [ ]:
ps['grp'] = ps.scheme_code.where(ps.scheme_code.isin(['NA', '-', '']), 'real code').replace({'': '(blank)'})
print(ps.groupby('grp').agg(rows=('free_qty', 'size'), with_free_goods=('free_qty', lambda x: int((x > 0).sum()))))
print('\ntop real codes:', ps[ps.grp == 'real code'].scheme_code.value_counts().head(5).to_dict())

**Check:** (blank) 48,446, `-` 16,294, `NA` 16,347 with **zero** free-goods rows each; real codes 12,722 with 11,023 free-goods rows. This supports, but does not prove, that the three mean no scheme. Confirm with the client.

## E5. Stock snapshot: conflicting quantities, row counts per C&F, stock-out markers

In [ ]:
sn = R['cnf_stock_snapshot'].copy()
sn['cnf'] = sn.cnf_code.str.replace(r'^CF(?=[A-Z]\d)', 'CF-', regex=True)
sn['qty'] = num(sn.qty_on_hand_packs)
print('rows per C&F (after fixing CFE01 style):\n', sn.cnf.value_counts().to_string())
hb = sn[sn.batch_no != '']
k = ['snapshot_date', 'cnf', 'sku_code', 'batch_no']
g = hb[hb.duplicated(k, keep=False)]
print('\nrows in conflicting groups:', len(g), ' groups:', g.groupby(k).ngroups)
print('groups where qty differs:', int((g.groupby(k).qty.nunique() > 1).sum()), ' where expiry differs:', int((g.groupby(k).expiry_date.nunique() > 1).sum()))
print('conflicting groups by C&F:\n', g.drop_duplicates(k).cnf.value_counts().to_string())
so = sn[(sn.batch_no == '') & (sn.qty == 0)]
print('\nblank-batch, qty 0 rows:', len(so))

**Check:** 831 conflicting groups (1,683 rows), quantity differs in all 831, expiry in none. Look at the C&F split: it tells us whether this is one site's habit. 270 blank-batch rows, all quantity 0.

## E6. Emails that chase credit notes (evidence for claim-processing delays)

In [ ]:
em = R['claim_emails']
hit = em[(em.subject + ' ' + em.body).str.contains('credit note|resend', case=False)]
for _, r in hit.iterrows():
    print(r.email_id, r.received_date, r.from_name, r.stockist_code_by_mailbox_rule, '|', r.subject, '|', r.body.replace('\r\n', ' ')[:200], '\n')

**Check:** EML-019 (Matrix Medicos, credit note not received yet), EML-005 (Ganesh Pharma Distributors, wants credit note), plus resends EML-022 and EML-021. Always cite email IDs like these, never from memory.